In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# FILE OVERVIEW: "Aam Zindagi" — building FAKE LangChain components to see WHY
#                the Runnable interface exists
# ─────────────────────────────────────────────────────────────────────────────
#
# WHAT THIS FILE DOES
#   It rebuilds LangChain's three core pieces from scratch as "Nakli" (fake/mock)
#   classes — a fake LLM, a fake PromptTemplate, and a fake Chain — using nothing
#   but plain Python OOP. No real model, no API. The goal is NOT generation; it's
#   to expose the STRUCTURE hiding behind prompt | model | parser.
#
# THE OOP LENS
#   Each component is a CLASS that encapsulates its own data + behaviour:
#     NakliLLM             -> knows how to .predict(prompt)
#     NakliPromptTemplate  -> knows how to .format(input_dict)
#     NakliLLMChain        -> knows how to .run(input_dict) by WIRING the other two
#   This is exactly how the real library is organised: separate objects, each with
#   one job, composed together.
#
# THE KEY OBSERVATION (this is the whole point — "aam zindagi" = the naive stage)
#   Look at the method names: predict(), format(), run(). Every class invents its
#   OWN method name. There is NO shared interface. So the chain (NakliLLMChain) is
#   forced to hard-code each component's specific method:
#       self.prompt.format(...)   then   self.llm.predict(...)
#   Because the names differ, you CANNOT treat these objects interchangeably, and
#   you cannot cleanly compose them with a single operator. The chain must know
#   the exact API of every part it touches.
#
# WHY THIS MATTERS — what it sets up next ("mafia zindagi")
#   The fix is a shared interface: define one abstract base class (Runnable) with
#   ONE standard method (invoke), and make EVERY component inherit it and
#   implement invoke() in its own way (polymorphism). Once LLM, PromptTemplate,
#   and Parser all speak the same invoke() language, a chain can call them
#   uniformly — and that uniformity is precisely what makes the | pipe possible.
#   So this file is the "before": it proves the problem that Runnables solve.
#
# CONCEPTS ON DISPLAY: classes, __init__ / encapsulation, methods as interfaces,
#   manual composition/wiring, and the ABSENCE of a common interface (the gap that
#   the Runnable abstract base class later fills).
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import random

class NakliLLM:                              # a MOCK large language model

  def __init__(self):
    print('LLM created')                     # runs when the object is built (proof of construction)

  def predict(self, prompt):                 # <-- this class's interface method is named "predict"
    # a real LLM would use `prompt` to generate text; this mock IGNORES it and
    # just returns a random canned answer — because we care about the STRUCTURE
    # (there is a predict() that takes a prompt and returns a dict), not the output
    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]
    return {'response': random.choice(response_list)}   # returns a DICT, mimicking a real API response shape

In [3]:
class NakliPromptTemplate:                   # a MOCK PromptTemplate

  def __init__(self, template, input_variables):
    self.template = template                 # store the template string, e.g. 'Write a {length} poem about {topic}'
    self.input_variables = input_variables   # store which placeholders it expects

  def format(self, input_dict):              # <-- this class's interface method is named "format" (DIFFERENT name again)
    # fills the {placeholders} in the template using the dict's values
    return self.template.format(**input_dict)   # **input_dict spreads {'length':'short',...} into the .format() call

In [4]:
template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',   # two placeholders
    input_variables=['length', 'topic']               # declared inputs
)
# nothing is filled yet — this just holds the template definition

In [5]:
prompt = template.format({'length': 'short', 'topic': 'india'})
# calls NakliPromptTemplate.format -> 'Write a short poem about india'
# NOTE: you had to CALL .format() by name. Step 1 of manual wiring.

In [6]:
llm = NakliLLM()        # prints 'LLM created' (the __init__ side effect)

LLM created


In [7]:
llm.predict(prompt)     # returns e.g. {'response': 'AI stands for Artificial Intelligence'}
# NOTE: you called .predict() by name — a DIFFERENT method name than .format().
# This is the friction: every step uses its own method, so YOU wire them by hand.

{'response': 'Delhi is the capital of India'}

In [8]:
class NakliLLMChain:                         # a MOCK Chain — its job is to WIRE prompt + llm together

  def __init__(self, llm, prompt):
    self.llm = llm                           # hold the two components it will connect
    self.prompt = prompt

  def run(self, input_dict):                 # <-- yet ANOTHER interface name: "run"
    # the chain manually performs the two steps you did by hand above:
    final_prompt = self.prompt.format(input_dict)   # step 1: format the prompt (calls .format)
    result = self.llm.predict(final_prompt)         # step 2: send it to the LLM (calls .predict)
    return result['response']                       # step 3: unwrap the dict -> plain string
    # THE LESSON: run() has to know EACH component's exact method (.format, .predict).
    # Because there's no shared interface, the chain is tightly coupled to those
    # specific names. This manual .format()->.predict()->unwrap is exactly what the
    # real `prompt | model | parser` pipe automates once everything is a Runnable.

In [9]:
template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

In [10]:
llm = NakliLLM()        # prints 'LLM created' again

LLM created


In [11]:
chain = NakliLLMChain(llm, template)   # inject both components into the chain (composition)

In [12]:
chain.run({'length': 'short', 'topic': 'india'})
# returns e.g. 'AI stands for Artificial Intelligence'
# ONE call now hides the format->predict->unwrap sequence. The chain encapsulated
# the wiring — but it only works because run() hard-codes each component's method.
# Standardise those methods into a single invoke() (the Runnable interface) and the
# chain no longer needs to know the specifics — that's the leap this file motivates.

'AI stands for Artificial Intelligence'